# 04 — Data Quality Bronze

**Credit Risk Intelligence Platform** — Camada Bronze

Este notebook implementa uma camada completa de **Data Quality** para as 8 tabelas Bronze do projeto Credit Risk.

## Responsabilidades

| Notebook | Responsabilidade |
| --- | --- |
| `01_ingestao_bronze` | Ingestão dos dados |
| `02_metadados_bronze` | Descrição e governância dos dados |
| `03_auditoria_bronze` | Auditoria operacional e histórico da ingestão |
| `04_data_quality_bronze` | **Qualidade, validações e métricas dos dados** (este notebook) |

## Dimensões de Data Quality implementadas

1. **Completeness** — NULLs e percentual de preenchimento
2. **Uniqueness** — Distintos, duplicidades e chaves candidatas
3. **Validity** — Validação de tipos e domínio de valores categóricos
4. **Consistency** — Regras de negócio do Home Credit
5. **Integrity** — Integridade referencial entre tabelas
6. **Outliers** — Detecção estatística via IQR
7. **Numeric Stats** — Estatísticas descritivas de colunas-chave
8. **TARGET Distribution** — Distribuição e desbalanceamento
9. **Quality Score** — Score agregado (0–100) com status PASS/WARNING/FAIL

## Importante

> Este notebook **NÃO modifica** os dados Bronze. Todas as verificações são de análise e registro apenas.
> Tratamentos (NULLs, duplicidades, outliers, tipos) serão realizados na camada Silver.

## Persistência

- `credit_risk.bronze.data_quality` — Detalhado de todas as métricas (histórico via `mode("append")`)
- `credit_risk.bronze.data_quality_summary` — Resumo por tabela e execução
- `credit_risk.bronze.data_quality_audit` — Auditoria da própria execução

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração, Parâmetros e Definições
# ============================================================================
# Imports
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid
import traceback

# ============================================================================
# THRESHOLDS DE DATA QUALITY (configuráveis)
# ============================================================================
DQ_THRESHOLDS = {
    "pass": 95,       # Score >= 95 → PASS
    "warning": 80,    # 80 <= Score < 95 → WARNING; Score < 80 → FAIL
}

# Thresholds específicos por dimensão
DIMENSION_THRESHOLDS = {
    "completeness": {"pass": 95.0, "warning": 80.0},   # % preenchimento
    "uniqueness": {"pass": 99.0, "warning": 90.0},      # % não-duplicidade (para chaves)
    "validity": {"pass": 98.0, "warning": 90.0},        # % conformidade de domínio
    "consistency": {"pass": 95.0, "warning": 85.0},      # % regras de negócio atendidas
    "integrity": {"pass": 99.0, "warning": 95.0},       # % integridade referencial
}

# ============================================================================
# METADADOS DA EXECUÇÃO
# ============================================================================
EXECUTION_TIMESTAMP = datetime.now(timezone.utc).isoformat()
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"dq_bronze_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_NAME = "04_data_quality_bronze"
NOTEBOOK_PATH = "/Users/abraaojose.100@gmail.com/Projeto_classificação/Projeto_Credit_Risk/04_data_quality_bronze"

# Acumulador global de resultados (lista de dicionários)
ALL_RESULTS = []

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🕐 Timestamp: {EXECUTION_TIMESTAMP}")

# ============================================================================
# LISTA DE TABELAS BRONZE
# ============================================================================
BRONZE_TABLES = [
    "credit_risk.bronze.application_train",
    "credit_risk.bronze.application_test",
    "credit_risk.bronze.bureau",
    "credit_risk.bronze.bureau_balance",
    "credit_risk.bronze.credit_card_balance",
    "credit_risk.bronze.installments_payments",
    "credit_risk.bronze.pos_cash_balance",
    "credit_risk.bronze.previous_application",
]

# ============================================================================
# CHAVES CANDIDATAS POR TABELA (para verificação de unicidade)
# ============================================================================
PRIMARY_KEY_CANDIDATES = {
    "application_train": ["SK_ID_CURR"],
    "application_test": ["SK_ID_CURR"],
    "bureau": ["SK_ID_BUREAU"],
    "bureau_balance": ["SK_ID_BUREAU", "MONTHS_BALANCE"],
    "credit_card_balance": ["SK_ID_PREV", "MONTHS_BALANCE"],
    "installments_payments": ["SK_ID_PREV", "NUM_INSTALMENT_NUMBER"],
    "pos_cash_balance": ["SK_ID_PREV", "MONTHS_BALANCE"],
    "previous_application": ["SK_ID_PREV"],
}

# Colunas de metadados a ignorar nas verificações de DQ
META_COLUMNS = {"_ingestion_timestamp", "_source_file"}

# ============================================================================
# DOMÍNIOS ESPERADOS PARA COLUNAS CATEGÓRICAS (validação de domínio)
# ============================================================================
# Definidos a partir dos valores reais observados nas tabelas Bronze.
# Valores fora deste conjunto serão registrados como "out_of_domain".
EXPECTED_DOMAINS = {
    # application_train / application_test
    ("application_train", "NAME_CONTRACT_TYPE"): ["Cash loans", "Revolving loans"],
    ("application_train", "CODE_GENDER"): ["F", "M", "XNA"],
    ("application_train", "FLAG_OWN_CAR"): ["Y", "N"],
    ("application_train", "FLAG_OWN_REALTY"): ["Y", "N"],
    ("application_train", "NAME_INCOME_TYPE"): ["Working", "State servant", "Commercial associate", "Pensioner", "Unemployed", "Student", "Businessman", "Maternity leave"],
    ("application_train", "NAME_EDUCATION_TYPE"): ["Secondary / secondary special", "Higher education", "Incomplete higher", "Lower secondary", "Academic degree"],
    ("application_train", "NAME_FAMILY_STATUS"): ["Single / not married", "Married", "Civil marriage", "Widow", "Separated", "Unknown"],
    ("application_train", "NAME_HOUSING_TYPE"): ["House / apartment", "Rented apartment", "With parents", "Municipal apartment", "Office apartment", "Co-op apartment"],
    ("application_train", "WEEKDAY_APPR_PROCESS_START"): ["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY", "SUNDAY"],
    ("application_train", "FONDKAPREMONT_MODE"): ["reg oper account", "reg oper spec account", "org spec account", "not specified"],
    ("application_train", "HOUSETYPE_MODE"): ["block of flats", "specific housing", "terraced house"],
    ("application_train", "WALLSMATERIAL_MODE"): ["Panel", "Stone, brick", "Block", "Mixed", "Wooden", "Monolithic", "Others"],
    ("application_train", "EMERGENCYSTATE_MODE"): ["Yes", "No"],
    # application_test herda os mesmos domínios da application_train
    ("application_test", "NAME_CONTRACT_TYPE"): ["Cash loans", "Revolving loans"],
    ("application_test", "CODE_GENDER"): ["F", "M", "XNA"],
    ("application_test", "FLAG_OWN_CAR"): ["Y", "N"],
    ("application_test", "FLAG_OWN_REALTY"): ["Y", "N"],
    ("application_test", "NAME_INCOME_TYPE"): ["Working", "State servant", "Commercial associate", "Pensioner", "Unemployed", "Student", "Businessman", "Maternity leave"],
    ("application_test", "NAME_EDUCATION_TYPE"): ["Secondary / secondary special", "Higher education", "Incomplete higher", "Lower secondary", "Academic degree"],
    ("application_test", "NAME_FAMILY_STATUS"): ["Single / not married", "Married", "Civil marriage", "Widow", "Separated", "Unknown"],
    ("application_test", "NAME_HOUSING_TYPE"): ["House / apartment", "Rented apartment", "With parents", "Municipal apartment", "Office apartment", "Co-op apartment"],
    # bureau
    ("bureau", "CREDIT_ACTIVE"): ["Active", "Closed", "Sold", "Bad debt"],
    ("bureau", "CREDIT_CURRENCY"): ["currency 1", "currency 2", "currency 3", "currency 4"],
    ("bureau", "CREDIT_TYPE"): ["Consumer credit", "Credit card", "Mortgage", "Car loan", "Microloan", "Loan for business development", "Loan for working capital replenishment", "Real estate loan", "Unknown type of loan", "Another type of loan", "Cash loan (non-earmarked)", "Loan for the purchase of equipment", "Mobile operator loan", "Interbank credit", "Loan for purchase of shares (margin lending)"],
    # bureau_balance
    ("bureau_balance", "STATUS"): ["C", "0", "X", "1", "2", "3", "4", "5"],
    # credit_card_balance
    ("credit_card_balance", "NAME_CONTRACT_STATUS"): ["Active", "Completed", "Demand", "Signed", "Sent proposal", "Approved", "Refused"],
    # pos_cash_balance
    ("pos_cash_balance", "NAME_CONTRACT_STATUS"): ["Active", "Completed", "Signed", "Demand", "Returned to the store", "Approved", "Canceled", "Amortized debt", "XNA"],
    # previous_application
    ("previous_application", "NAME_CONTRACT_TYPE"): ["Cash loans", "Consumer loans", "Revolving loans", "XNA"],
    ("previous_application", "NAME_CONTRACT_STATUS"): ["Approved", "Canceled", "Refused", "Unused offer"],
    ("previous_application", "NAME_PAYMENT_TYPE"): ["Cash through the bank", "Non-cash from your account", "Cashless from the account of the employer", "XNA"],
    ("previous_application", "NAME_CLIENT_TYPE"): ["New", "Repeater", "Refreshed", "XNA"],
    ("previous_application", "NAME_PORTFOLIO"): ["POS", "Cash", "Cards", "Cars", "XNA"],
    ("previous_application", "NAME_PRODUCT_TYPE"): ["x-sell", "walk-in", "XNA"],
    ("previous_application", "NAME_YIELD_GROUP"): ["high", "middle", "low_normal", "low_action", "XNA"],
}

# ============================================================================
# REGRAS DE NEGÓCIO DO HOME CREDIT (validação de consistência)
# ============================================================================
# Formato: (table_short_name, rule_name, rule_description, column_name, condition_sql, threshold_pct)
# condition_sql: condição que define uma FALHA (registros que violam a regra)
BUSINESS_RULES = {
    "application_train": [
        {"rule_name": "AMT_INCOME_TOTAL_POSITIVE", "rule_desc": "AMT_INCOME_TOTAL deve ser > 0", "column": "AMT_INCOME_TOTAL", "fail_condition": "AMT_INCOME_TOTAL <= 0 OR AMT_INCOME_TOTAL IS NULL"},
        {"rule_name": "AMT_CREDIT_POSITIVE", "rule_desc": "AMT_CREDIT deve ser > 0", "column": "AMT_CREDIT", "fail_condition": "AMT_CREDIT <= 0 OR AMT_CREDIT IS NULL"},
        {"rule_name": "AMT_ANNUITY_NON_NEGATIVE", "rule_desc": "AMT_ANNUITY deve ser >= 0", "column": "AMT_ANNUITY", "fail_condition": "AMT_ANNUITY < 0"},
        {"rule_name": "AMT_GOODS_PRICE_NON_NEGATIVE", "rule_desc": "AMT_GOODS_PRICE deve ser >= 0", "column": "AMT_GOODS_PRICE", "fail_condition": "AMT_GOODS_PRICE < 0"},
        {"rule_name": "CNT_CHILDREN_NON_NEGATIVE", "rule_desc": "CNT_CHILDREN deve ser >= 0", "column": "CNT_CHILDREN", "fail_condition": "CNT_CHILDREN < 0"},
        {"rule_name": "DAYS_BIRTH_NEGATIVE", "rule_desc": "DAYS_BIRTH deve ser negativo (idade relativa à aplicação)", "column": "DAYS_BIRTH", "fail_condition": "DAYS_BIRTH >= 0"},
        {"rule_name": "DAYS_EMPLOYED_NEGATIVE", "rule_desc": "DAYS_EMPLOYED deve ser negativo (tempo empregado)", "column": "DAYS_EMPLOYED", "fail_condition": "DAYS_EMPLOYED >= 0 AND DAYS_EMPLOYED != 365243"},
        {"rule_name": "TARGET_BINARY", "rule_desc": "TARGET deve ser 0 ou 1", "column": "TARGET", "fail_condition": "TARGET NOT IN (0, 1)"},
    ],
    "application_test": [
        {"rule_name": "AMT_INCOME_TOTAL_POSITIVE", "rule_desc": "AMT_INCOME_TOTAL deve ser > 0", "column": "AMT_INCOME_TOTAL", "fail_condition": "AMT_INCOME_TOTAL <= 0 OR AMT_INCOME_TOTAL IS NULL"},
        {"rule_name": "AMT_CREDIT_POSITIVE", "rule_desc": "AMT_CREDIT deve ser > 0", "column": "AMT_CREDIT", "fail_condition": "AMT_CREDIT <= 0 OR AMT_CREDIT IS NULL"},
        {"rule_name": "CNT_CHILDREN_NON_NEGATIVE", "rule_desc": "CNT_CHILDREN deve ser >= 0", "column": "CNT_CHILDREN", "fail_condition": "CNT_CHILDREN < 0"},
    ],
    "bureau": [
        {"rule_name": "AMT_CREDIT_SUM_NON_NEGATIVE", "rule_desc": "AMT_CREDIT_SUM deve ser >= 0", "column": "AMT_CREDIT_SUM", "fail_condition": "AMT_CREDIT_SUM < 0"},
        {"rule_name": "DAYS_CREDIT_NEGATIVE", "rule_desc": "DAYS_CREDIT deve ser <= 0 (dias antes da aplicação)", "column": "DAYS_CREDIT", "fail_condition": "DAYS_CREDIT > 0"},
    ],
    "previous_application": [
        {"rule_name": "AMT_CREDIT_NON_NEGATIVE", "rule_desc": "AMT_CREDIT deve ser >= 0", "column": "AMT_CREDIT", "fail_condition": "AMT_CREDIT < 0"},
        {"rule_name": "AMT_APPLICATION_NON_NEGATIVE", "rule_desc": "AMT_APPLICATION deve ser >= 0", "column": "AMT_APPLICATION", "fail_condition": "AMT_APPLICATION < 0"},
        {"rule_name": "AMT_ANNUITY_NON_NEGATIVE", "rule_desc": "AMT_ANNUITY deve ser >= 0", "column": "AMT_ANNUITY", "fail_condition": "AMT_ANNUITY < 0"},
        {"rule_name": "DAYS_DECISION_NEGATIVE", "rule_desc": "DAYS_DECISION deve ser <= 0", "column": "DAYS_DECISION", "fail_condition": "DAYS_DECISION > 0"},
    ],
    "installments_payments": [
        {"rule_name": "AMT_INSTALMENT_NON_NEGATIVE", "rule_desc": "AMT_INSTALMENT deve ser >= 0", "column": "AMT_INSTALMENT", "fail_condition": "AMT_INSTALMENT < 0"},
        {"rule_name": "AMT_PAYMENT_NON_NEGATIVE", "rule_desc": "AMT_PAYMENT deve ser >= 0", "column": "AMT_PAYMENT", "fail_condition": "AMT_PAYMENT < 0"},
    ],
}

# ============================================================================
# MAPEAMENTOS DE INTEGRIDADE REFERENCIAL
# ============================================================================
# (parent_table, child_table, parent_key, child_key)
REFERENTIAL_INTEGRITY_MAPPINGS = [
    ("application_train", "bureau", "SK_ID_CURR", "SK_ID_CURR"),
    ("application_train", "previous_application", "SK_ID_CURR", "SK_ID_CURR"),
    ("application_train", "pos_cash_balance", "SK_ID_CURR", "SK_ID_CURR"),
    ("application_train", "credit_card_balance", "SK_ID_CURR", "SK_ID_CURR"),
    ("application_train", "installments_payments", "SK_ID_CURR", "SK_ID_CURR"),
    # SK_ID_PREV entre tabelas
    ("previous_application", "credit_card_balance", "SK_ID_PREV", "SK_ID_PREV"),
    ("previous_application", "pos_cash_balance", "SK_ID_PREV", "SK_ID_PREV"),
    ("previous_application", "installments_payments", "SK_ID_PREV", "SK_ID_PREV"),
    # SK_ID_BUREAU entre tabelas
    ("bureau", "bureau_balance", "SK_ID_BUREAU", "SK_ID_BUREAU"),
]

# ============================================================================
# COLUNAS NUMÉRICAS PRIORITÁRIAS PARA ESTATÍSTICAS E OUTLIERS
# ============================================================================
# Definidas para evitar custo computacional desnecessário em tabelas com 120+ colunas.
NUMERIC_PRIORITY_COLUMNS = {
    "application_train": ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_CHILDREN", "DAYS_BIRTH", "DAYS_EMPLOYED", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE"],
    "application_test": ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_CHILDREN", "DAYS_BIRTH", "DAYS_EMPLOYED", "CNT_FAM_MEMBERS"],
    "bureau": ["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_MAX_OVERDUE", "AMT_ANNUITY", "DAYS_CREDIT", "DAYS_CREDIT_ENDDATE"],
    "bureau_balance": ["MONTHS_BALANCE"],
    "credit_card_balance": ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT", "MONTHS_BALANCE"],
    "installments_payments": ["AMT_INSTALMENT", "AMT_PAYMENT", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "NUM_INSTALMENT_NUMBER"],
    "pos_cash_balance": ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE", "MONTHS_BALANCE"],
    "previous_application": ["AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_GOODS_PRICE", "AMT_DOWN_PAYMENT", "DAYS_DECISION", "CNT_PAYMENT"],
}

print("✅ Configuração carregada com sucesso!")
print(f"   Tabelas Bronze: {len(BRONZE_TABLES)}")
print(f"   Domínios esperados: {len(EXPECTED_DOMAINS)}")
print(f"   Regras de negócio: {sum(len(v) for v in BUSINESS_RULES.values())}")
print(f"   Mapeamentos RI: {len(REFERENTIAL_INTEGRITY_MAPPINGS)}")

In [0]:
# ============================================================================
# CÉLULA 2 — Funções Auxiliares Modulares
# ============================================================================
# Cada função implementa uma dimensão de Data Quality e retorna uma lista
# de dicionários (rows) que serão acumulados em ALL_RESULTS.


def get_table_short_name(full_table_name):
    """Extrai o nome curto da tabela (ex: credit_risk.bronze.application_train → application_train)."""
    return full_table_name.split(".")[-1]


def _make_base_row(table_name, quality_dimension, metric_name, metric_value,
                   column_name="__TABLE__", threshold=None, status="PASS",
                   rule_name=None, rule_description=None, parent_table=None,
                   child_table=None, parent_key=None, child_key=None):
    """Cria um dicionário base padronizado para todas as métricas de DQ."""
    # Tenta converter metric_value para float; se não for numérico, armazena como None
    # e preserva o valor original em rule_description.
    if metric_value is not None:
        try:
            metric_value_float = float(metric_value)
        except (ValueError, TypeError):
            metric_value_float = None
            extra = f"value={metric_value}"
            rule_description = f"{rule_description}; {extra}" if rule_description else extra
    else:
        metric_value_float = None
    return {
        "execution_timestamp": EXECUTION_TIMESTAMP,
        "execution_id": EXECUTION_ID,
        "batch_id": BATCH_ID,
        "table_name": table_name,
        "column_name": column_name,
        "quality_dimension": quality_dimension,
        "metric_name": metric_name,
        "metric_value": metric_value_float,
        "threshold": float(threshold) if threshold is not None else None,
        "status": status,
        "rule_name": rule_name,
        "rule_description": rule_description,
        "parent_table": parent_table,
        "child_table": child_table,
        "parent_key": parent_key,
        "child_key": child_key,
    }


def get_column_types(df):
    """Retorna um dicionário {column_name: data_type_string} a partir do schema do DataFrame."""
    return {f.name: f.dataType.simpleString() for f in df.schema.fields}


def get_data_columns(df):
    """Retorna lista de colunas de dados (excluindo colunas de metadados de ingestão)."""
    return [f.name for f in df.schema.fields if f.name not in META_COLUMNS]


# ----------------------------------------------------------------------------
# 1. COMPLETENESS — NULLs e percentual de preenchimento
# ----------------------------------------------------------------------------
def run_completeness_check(table_full_name, df, row_count):
    """
    Calcula para cada coluna: null_count, null_percentage, completeness_percentage.
    Usa uma única agregação Spark para todas as colunas (evita múltiplas passagens).
    """
    results = []
    short_name = get_table_short_name(table_full_name)
    data_cols = get_data_columns(df)

    if row_count == 0:
        for col_name in data_cols:
            results.append(_make_base_row(table_full_name, "Completeness", "null_count", 0, col_name))
            results.append(_make_base_row(table_full_name, "Completeness", "null_percentage", 100.0, col_name, threshold=DIMENSION_THRESHOLDS["completeness"]["pass"], status="FAIL"))
            results.append(_make_base_row(table_full_name, "Completeness", "completeness_percentage", 0.0, col_name, threshold=DIMENSION_THRESHOLDS["completeness"]["pass"], status="FAIL"))
        return results

    # Constrói uma única expressão de agregação para todas as colunas
    agg_exprs = []
    for col_name in data_cols:
        agg_exprs.append(F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias(f"__null_{col_name}"))

    # Executa uma única agregação (uma passagem sobre os dados)
    null_row = df.agg(*agg_exprs).collect()[0]

    for col_name in data_cols:
        null_count = null_row[f"__null_{col_name}"]
        null_pct = (null_count / row_count * 100.0) if row_count > 0 else 100.0
        completeness_pct = 100.0 - null_pct

        results.append(_make_base_row(table_full_name, "Completeness", "null_count", null_count, col_name))
        results.append(_make_base_row(table_full_name, "Completeness", "null_percentage", null_pct, col_name, threshold=DIMENSION_THRESHOLDS["completeness"]["pass"], status="PASS"))
        results.append(_make_base_row(table_full_name, "Completeness", "completeness_percentage", completeness_pct, col_name, threshold=DIMENSION_THRESHOLDS["completeness"]["pass"], status="PASS"))

    # Também registra o row_count a nível de tabela
    results.append(_make_base_row(table_full_name, "Completeness", "row_count", row_count))

    return results


# ----------------------------------------------------------------------------
# 2. UNIQUENESS — Distintos, duplicidades e chaves candidatas
# ----------------------------------------------------------------------------
def run_uniqueness_check(table_full_name, df, row_count):
    """
    Para cada chave candidata: distinct_count, duplicate_count, duplicate_percentage.
    Também verifica duplicidade completa de registros (hash de todas as colunas).
    """
    results = []
    short_name = get_table_short_name(table_full_name)
    key_cols = PRIMARY_KEY_CANDIDATES.get(short_name, [])

    # Verifica unicidade das chaves candidatas
    for key_col in key_cols:
        if key_col not in df.columns:
            continue

        distinct_count = df.select(key_col).distinct().count()
        duplicate_count = row_count - distinct_count
        duplicate_pct = (duplicate_count / row_count * 100.0) if row_count > 0 else 0.0
        uniqueness_pct = 100.0 - duplicate_pct

        status = "PASS" if uniqueness_pct >= DIMENSION_THRESHOLDS["uniqueness"]["pass"] else ("WARNING" if uniqueness_pct >= DIMENSION_THRESHOLDS["uniqueness"]["warning"] else "FAIL")

        results.append(_make_base_row(table_full_name, "Uniqueness", "distinct_count", distinct_count, key_col))
        results.append(_make_base_row(table_full_name, "Uniqueness", "duplicate_count", duplicate_count, key_col))
        results.append(_make_base_row(table_full_name, "Uniqueness", "duplicate_percentage", duplicate_pct, key_col, threshold=DIMENSION_THRESHOLDS["uniqueness"]["pass"], status=status))

        # Determina se a coluna é chave primária candidata
        is_pk = "YES" if duplicate_count == 0 and row_count > 0 else "NO"
        results.append(_make_base_row(table_full_name, "Uniqueness", "is_primary_key_candidate", is_pk, key_col, status="PASS" if is_pk == "YES" else "WARNING"))

    # Duplicidade completa de registros (usa hash MD5 de todas as colunas de dados)
    data_cols = get_data_columns(df)
    if len(data_cols) > 0 and row_count > 0:
        # Hash de todas as colunas para detectar registros idênticos
        hash_col = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in data_cols])
        full_dup_count = df.select(hash_col.alias("__row_hash")).groupBy("__row_hash").agg(F.count("*").alias("cnt")).filter("cnt > 1").agg(F.sum(F.col("cnt") - 1).alias("dup_total")).collect()[0]["dup_total"]
        full_dup_count = full_dup_count if full_dup_count is not None else 0
        full_dup_pct = (full_dup_count / row_count * 100.0) if row_count > 0 else 0.0

        status = "PASS" if full_dup_pct < 1.0 else ("WARNING" if full_dup_pct < 5.0 else "FAIL")
        results.append(_make_base_row(table_full_name, "Uniqueness", "full_duplicate_count", full_dup_count, "__TABLE__", status=status))
        results.append(_make_base_row(table_full_name, "Uniqueness", "full_duplicate_percentage", full_dup_pct, "__TABLE__", threshold=1.0, status=status))

    return results


# ----------------------------------------------------------------------------
# 3. TYPE VALIDATION — Validação de tipos do schema
# ----------------------------------------------------------------------------
def run_type_validation(table_full_name, df):
    """
    Registra o tipo atual de cada coluna. Na Bronze, o tipo observado é aceito como esperado
    (documentação). Em futuras execuções, mudanças de tipo serão detectadas.
    """
    results = []
    col_types = get_column_types(df)

    for col_name, col_type in col_types.items():
        # Na Bronze, o tipo atual é documentado; status = PASS (sem correção)
        results.append(_make_base_row(
            table_full_name, "Validity", "type_validation",
            col_type,  # metric_value armazena o tipo como string (usar field extra abaixo)
            column_name=col_name,
            status="PASS",
        ))
        # Sobrescreve metric_value com valor numérico não aplicável; armazenamos o tipo em rule_description
        results[-1]["metric_value"] = None
        results[-1]["rule_description"] = f"actual_data_type={col_type}"
        results[-1]["metric_name"] = "actual_data_type"

    return results


# ----------------------------------------------------------------------------
# 4. DOMAIN VALIDATION — Validação de domínio para colunas categóricas
# ----------------------------------------------------------------------------
def run_domain_validation(table_full_name, df, row_count):
    """
    Para cada coluna categórica com domínio definido:
    - Identifica valores distintos reais
    - Identifica valores NULL
    - Registra valores fora do domínio esperado
    """
    results = []
    short_name = get_table_short_name(table_full_name)

    for (table_short, col_name), expected_values in EXPECTED_DOMAINS.items():
        if table_short != short_name or col_name not in df.columns:
            continue

        # Coleta valores distintos reais (limite para evitar custo excessivo)
        actual_values = [row[col_name] for row in df.select(col_name).distinct().limit(200).collect()]
        actual_set = set(v for v in actual_values if v is not None)
        expected_set = set(expected_values)

        # Valores fora do domínio
        out_of_domain = actual_set - expected_set
        has_nulls = None in actual_values or any(v is None for v in actual_values)

        # Conta registros fora do domínio (Spark SQL para eficiência)
        expected_sql = ", ".join([f"'{v}'" for v in expected_values])
        out_of_domain_count = df.filter(
            (F.col(col_name).isNotNull()) & (~F.col(col_name).isin(expected_values))
        ).count()

        null_in_col = df.filter(F.col(col_name).isNull()).count()

        ood_pct = (out_of_domain_count / row_count * 100.0) if row_count > 0 else 0.0
        domain_conformity_pct = 100.0 - ood_pct

        status = "PASS" if domain_conformity_pct >= DIMENSION_THRESHOLDS["validity"]["pass"] else ("WARNING" if domain_conformity_pct >= DIMENSION_THRESHOLDS["validity"]["warning"] else "FAIL")

        results.append(_make_base_row(
            table_full_name, "Validity", "distinct_values_count", len(actual_set), col_name
        ))
        results.append(_make_base_row(
            table_full_name, "Validity", "null_values_in_category", null_in_col, col_name
        ))
        results.append(_make_base_row(
            table_full_name, "Validity", "out_of_domain_count", out_of_domain_count, col_name,
            threshold=DIMENSION_THRESHOLDS["validity"]["pass"], status=status,
            rule_description=f"Expected: {expected_values}; Out_of_domain: {list(out_of_domain)[:10]}"
        ))
        results.append(_make_base_row(
            table_full_name, "Validity", "domain_conformity_percentage", domain_conformity_pct, col_name,
            threshold=DIMENSION_THRESHOLDS["validity"]["pass"], status=status
        ))

    return results


# ----------------------------------------------------------------------------
# 5. BUSINESS RULES — Regras de negócio do Home Credit
# ----------------------------------------------------------------------------
def run_business_rules(table_full_name, df, row_count):
    """
    Executa cada regra de negócio definida para a tabela.
    Registra: failed_count, failed_percentage, rule_status.
    """
    results = []
    short_name = get_table_short_name(table_full_name)
    rules = BUSINESS_RULES.get(short_name, [])

    for rule in rules:
        col = rule["column"]
        if col not in df.columns:
            results.append(_make_base_row(
                table_full_name, "Consistency", "rule_not_applicable", 0, col,
                rule_name=rule["rule_name"], rule_description=rule["rule_desc"],
                status="WARNING"
            ))
            continue

        # Conta registros que violam a regra (Spark SQL para eficiência)
        failed_count = df.filter(rule["fail_condition"]).count()
        failed_pct = (failed_count / row_count * 100.0) if row_count > 0 else 0.0
        pass_pct = 100.0 - failed_pct

        status = "PASS" if pass_pct >= DIMENSION_THRESHOLDS["consistency"]["pass"] else ("WARNING" if pass_pct >= DIMENSION_THRESHOLDS["consistency"]["warning"] else "FAIL")

        results.append(_make_base_row(
            table_full_name, "Consistency", "business_rule_failed_count", failed_count, col,
            threshold=0, status=status,
            rule_name=rule["rule_name"], rule_description=rule["rule_desc"]
        ))
        results.append(_make_base_row(
            table_full_name, "Consistency", "business_rule_failed_percentage", failed_pct, col,
            threshold=DIMENSION_THRESHOLDS["consistency"]["pass"], status=status,
            rule_name=rule["rule_name"], rule_description=rule["rule_desc"]
        ))

    return results


# ----------------------------------------------------------------------------
# 6. REFERENTIAL INTEGRITY — Integridade referencial entre tabelas
# ----------------------------------------------------------------------------
def run_referential_integrity(tables_cache):
    """
    Para cada mapeamento RI, verifica órfãos no lado child.
    tables_cache: dict {table_short_name: DataFrame}
    """
    results = []

    for parent_short, child_short, parent_key, child_key in REFERENTIAL_INTEGRITY_MAPPINGS:
        parent_df = tables_cache.get(parent_short)
        child_df = tables_cache.get(child_short)

        if parent_df is None or child_df is None:
            results.append(_make_base_row(
                f"credit_risk.bronze.{child_short}", "Integrity", "ri_not_checked", 0,
                child_key, status="WARNING",
                rule_name=f"RI_{parent_short}_{child_short}",
                rule_description=f"Tabela {parent_short} ou {child_short} não disponível",
                parent_table=parent_short, child_table=child_short,
                parent_key=parent_key, child_key=child_key
            ))
            continue

        if parent_key not in parent_df.columns or child_key not in child_df.columns:
            results.append(_make_base_row(
                f"credit_risk.bronze.{child_short}", "Integrity", "ri_column_missing", 0,
                child_key, status="WARNING",
                rule_name=f"RI_{parent_short}_{child_short}",
                rule_description=f"Coluna {parent_key} ou {child_key} não existe",
                parent_table=parent_short, child_table=child_short,
                parent_key=parent_key, child_key=child_key
            ))
            continue

        # Conta órfãos: registros no child que não têm correspondência no parent
        # Usa LEFT ANTI JOIN para eficiência (não traz dados para o driver)
        child_count = child_df.count()
        orphan_count = child_df.join(
            parent_df.select(parent_key).distinct(),
            on=child_key,
            how="left_anti"
        ).count()

        orphan_pct = (orphan_count / child_count * 100.0) if child_count > 0 else 0.0
        ri_pct = 100.0 - orphan_pct

        status = "PASS" if ri_pct >= DIMENSION_THRESHOLDS["integrity"]["pass"] else ("WARNING" if ri_pct >= DIMENSION_THRESHOLDS["integrity"]["warning"] else "FAIL")

        results.append(_make_base_row(
            f"credit_risk.bronze.{child_short}", "Integrity", "orphan_count", orphan_count, child_key,
            status=status, rule_name=f"RI_{parent_short}_{child_short}",
            rule_description=f"Child {child_short}.{child_key} → Parent {parent_short}.{parent_key}",
            parent_table=parent_short, child_table=child_short,
            parent_key=parent_key, child_key=child_key
        ))
        results.append(_make_base_row(
            f"credit_risk.bronze.{child_short}", "Integrity", "orphan_percentage", orphan_pct, child_key,
            threshold=DIMENSION_THRESHOLDS["integrity"]["pass"], status=status,
            rule_name=f"RI_{parent_short}_{child_short}",
            rule_description=f"Child {child_short}.{child_key} → Parent {parent_short}.{parent_key}",
            parent_table=parent_short, child_table=child_short,
            parent_key=parent_key, child_key=child_key
        ))
        results.append(_make_base_row(
            f"credit_risk.bronze.{child_short}", "Integrity", "referential_integrity_percentage", ri_pct, child_key,
            threshold=DIMENSION_THRESHOLDS["integrity"]["pass"], status=status,
            rule_name=f"RI_{parent_short}_{child_short}",
            rule_description=f"Child {child_short}.{child_key} → Parent {parent_short}.{parent_key}",
            parent_table=parent_short, child_table=child_short,
            parent_key=parent_key, child_key=child_key
        ))

    return results


# ----------------------------------------------------------------------------
# 7. NUMERIC STATS — Estatísticas descritivas de colunas numéricas prioritárias
# ----------------------------------------------------------------------------
def run_numeric_stats(table_full_name, df, row_count):
    """
    Calcula min, max, mean, stddev, percentis (25, 50, 75) para colunas numéricas prioritárias.
    Usa percentile_approx para eficiência (aproximação via GDNA).
    """
    results = []
    short_name = get_table_short_name(table_full_name)
    priority_cols = NUMERIC_PRIORITY_COLUMNS.get(short_name, [])

    numeric_types = {"int", "bigint", "double", "float", "decimal", "long", "short"}

    for col_name in priority_cols:
        if col_name not in df.columns:
            continue
        col_type = dict((f.name, f.dataType.simpleString()) for f in df.schema.fields).get(col_name, "")
        if col_type not in numeric_types:
            continue

        # Usa Spark agg para computar todas as estatísticas em uma única passagem
        stats_row = df.agg(
            F.min(col_name).alias("min_val"),
            F.max(col_name).alias("max_val"),
            F.mean(col_name).alias("mean_val"),
            F.stddev(col_name).alias("stddev_val"),
            F.percentile_approx(col_name, 0.25).alias("p25"),
            F.percentile_approx(col_name, 0.50).alias("p50"),
            F.percentile_approx(col_name, 0.75).alias("p75"),
        ).collect()[0]

        # median = p50
        stats_map = {
            "min_value": stats_row["min_val"],
            "max_value": stats_row["max_val"],
            "mean_value": stats_row["mean_val"],
            "stddev_value": stats_row["stddev_val"],
            "percentile_25": stats_row["p25"],
            "percentile_50": stats_row["p50"],
            "percentile_75": stats_row["p75"],
        }

        for stat_name, stat_value in stats_map.items():
            results.append(_make_base_row(
                table_full_name, "NumericStats", stat_name, stat_value, col_name
            ))

    return results


# ----------------------------------------------------------------------------
# 8. OUTLIER DETECTION — Detecção de outliers via IQR
# ----------------------------------------------------------------------------
def run_outlier_detection(table_full_name, df, row_count):
    """
    Identifica outliers usando o método IQR (Q1 - 1.5*IQR, Q3 + 1.5*IQR).
    Registra: q1, q3, iqr, lower_bound, upper_bound, outlier_count, outlier_percentage.
    """
    results = []
    short_name = get_table_short_name(table_full_name)
    priority_cols = NUMERIC_PRIORITY_COLUMNS.get(short_name, [])
    numeric_types = {"int", "bigint", "double", "float", "decimal", "long", "short"}

    for col_name in priority_cols:
        if col_name not in df.columns:
            continue
        col_type = dict((f.name, f.dataType.simpleString()) for f in df.schema.fields).get(col_name, "")
        if col_type not in numeric_types:
            continue

        # Computa Q1, Q3 e IQR em uma única passagem
        pct_row = df.agg(
            F.percentile_approx(col_name, 0.25).alias("q1"),
            F.percentile_approx(col_name, 0.75).alias("q3"),
        ).collect()[0]

        q1 = pct_row["q1"]
        q3 = pct_row["q3"]
        iqr = (q3 - q1) if (q1 is not None and q3 is not None) else None

        if iqr is not None:
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            outlier_count = df.filter(
                (F.col(col_name) < lower_bound) | (F.col(col_name) > upper_bound)
            ).count()
            outlier_pct = (outlier_count / row_count * 100.0) if row_count > 0 else 0.0

            status = "PASS" if outlier_pct < 5.0 else ("WARNING" if outlier_pct < 15.0 else "FAIL")
        else:
            lower_bound = None
            upper_bound = None
            outlier_count = 0
            outlier_pct = 0.0
            status = "PASS"

        results.append(_make_base_row(table_full_name, "Outliers", "q1", q1, col_name))
        results.append(_make_base_row(table_full_name, "Outliers", "q3", q3, col_name))
        results.append(_make_base_row(table_full_name, "Outliers", "iqr", iqr, col_name))
        results.append(_make_base_row(table_full_name, "Outliers", "lower_bound", lower_bound, col_name))
        results.append(_make_base_row(table_full_name, "Outliers", "upper_bound", upper_bound, col_name))
        results.append(_make_base_row(table_full_name, "Outliers", "outlier_count", outlier_count, col_name, status=status))
        results.append(_make_base_row(table_full_name, "Outliers", "outlier_percentage", outlier_pct, col_name, threshold=5.0, status=status))

    return results


# ----------------------------------------------------------------------------
# 9. TARGET DISTRIBUTION — Distribuição da variável TARGET
# ----------------------------------------------------------------------------
def run_target_distribution(table_full_name, df, row_count):
    """
    Analisa a distribuição de TARGET em application_train.
    Calcula class_count, class_percentage e identifica desbalanceamento.
    """
    results = []
    short_name = get_table_short_name(table_full_name)

    if short_name != "application_train" or "TARGET" not in df.columns:
        return results

    target_dist = df.groupBy("TARGET").count().orderBy("TARGET").collect()

    for row in target_dist:
        target_val = row["TARGET"]
        class_count = row["count"]
        class_pct = (class_count / row_count * 100.0) if row_count > 0 else 0.0

        results.append(_make_base_row(
            table_full_name, "TargetDistribution", "class_count", class_count, "TARGET",
            rule_name=f"TARGET_{target_val}",
            rule_description=f"TARGET={target_val}"
        ))
        results.append(_make_base_row(
            table_full_name, "TargetDistribution", "class_percentage", class_pct, "TARGET",
            rule_name=f"TARGET_{target_val}",
            rule_description=f"TARGET={target_val}"
        ))

    # Verifica desbalanceamento
    if len(target_dist) == 2:
        counts = [r["count"] for r in target_dist]
        total = sum(counts)
        minority_pct = min(counts) / total * 100.0 if total > 0 else 0.0
        imbalance_ratio = max(counts) / min(counts) if min(counts) > 0 else float("inf")

        status = "PASS" if minority_pct >= 20.0 else ("WARNING" if minority_pct >= 10.0 else "FAIL")

        results.append(_make_base_row(
            table_full_name, "TargetDistribution", "imbalance_ratio", imbalance_ratio, "TARGET",
            status=status,
            rule_description=f"Minority class = {minority_pct:.2f}% (ratio 1:{imbalance_ratio:.2f})"
        ))
        results.append(_make_base_row(
            table_full_name, "TargetDistribution", "minority_class_percentage", minority_pct, "TARGET",
            threshold=20.0, status=status,
            rule_description=f"Minority class percentage (threshold 20%)"
        ))

    return results


print("✅ Funções auxiliares definidas!")
print("   9 funções modulares: completeness, uniqueness, type_validation, domain_validation,")
print("   business_rules, referential_integrity, numeric_stats, outlier_detection, target_distribution")

In [0]:
# ============================================================================
# CÉLULA 3 — Execução Principal: Run All DQ Checks
# ============================================================================
# Itera sobre todas as 8 tabelas Bronze executando cada dimensão de DQ.
# Cada check é envolvido em try/except individual: se um check falhar,
# o erro é registrado e os demais checks continuam.

EXECUTION_START = datetime.now(timezone.utc)
TABLE_ERRORS = []
TABLES_CACHE = {}  # Cache de DataFrames para reuso na verificação de integridade referencial


def _safe_run(check_name, func, *args, **kwargs):
    """Executa uma função de DQ de forma segura. Erros são registrados mas não interrompem a execução."""
    try:
        return func(*args, **kwargs)
    except Exception as e:
        error_msg = str(e)[:300]
        TABLE_ERRORS.append({"table": get_table_short_name(args[0]), "check": check_name, "error": error_msg})
        print(f"      ⚠️ {check_name} falhou: {error_msg[:120]}")
        return [_make_base_row(args[0], "Execution", f"{check_name}_error", 1, status="FAIL", rule_description=error_msg)]


print("=" * 80)
print("INICIANDO EXECUÇÃO DE DATA QUALITY — BRONZE")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Batch ID: {BATCH_ID}")
print("=" * 80)

for table_full_name in BRONZE_TABLES:
    short_name = get_table_short_name(table_full_name)
    table_start = datetime.now(timezone.utc)

    print(f"\n{'─' * 60}")
    print(f"📊 Analisando: {short_name}")
    print(f"{'─' * 60}")

    try:
        df = spark.table(table_full_name)
        row_count = df.count()
        TABLES_CACHE[short_name] = df
        print(f"   Row count: {row_count:,}")
    except Exception as e:
        error_msg = str(e)[:500]
        TABLE_ERRORS.append({"table": short_name, "check": "load_table", "error": error_msg})
        print(f"   ❌ ERRO ao carregar tabela: {error_msg}")
        ALL_RESULTS.append(_make_base_row(table_full_name, "Execution", "load_error", 1, status="FAIL", rule_description=error_msg))
        continue

    # Cada check é executado de forma independente (erro em um não afeta os demais)
    print("   ✅ Completeness...")
    ALL_RESULTS.extend(_safe_run("completeness", run_completeness_check, table_full_name, df, row_count))

    print("   ✅ Uniqueness...")
    ALL_RESULTS.extend(_safe_run("uniqueness", run_uniqueness_check, table_full_name, df, row_count))

    print("   ✅ Type Validation...")
    ALL_RESULTS.extend(_safe_run("type_validation", run_type_validation, table_full_name, df))

    print("   ✅ Domain Validation...")
    ALL_RESULTS.extend(_safe_run("domain_validation", run_domain_validation, table_full_name, df, row_count))

    print("   ✅ Business Rules...")
    ALL_RESULTS.extend(_safe_run("business_rules", run_business_rules, table_full_name, df, row_count))

    print("   ✅ Numeric Stats...")
    ALL_RESULTS.extend(_safe_run("numeric_stats", run_numeric_stats, table_full_name, df, row_count))

    print("   ✅ Outlier Detection...")
    ALL_RESULTS.extend(_safe_run("outlier_detection", run_outlier_detection, table_full_name, df, row_count))

    if short_name == "application_train":
        print("   ✅ TARGET Distribution...")
        ALL_RESULTS.extend(_safe_run("target_distribution", run_target_distribution, table_full_name, df, row_count))

    table_end = datetime.now(timezone.utc)
    duration = (table_end - table_start).total_seconds()
    print(f"   ⏱️ Concluído em {duration:.1f}s | Métricas acumuladas: {len(ALL_RESULTS)}")

# 6. REFERENTIAL INTEGRITY (executada após todas as tabelas serem carregadas)
print(f"\n{'─' * 60}")
print(f"🔗 Verificando Integridade Referencial...")
print(f"{'─' * 60}")
try:
    ri_results = run_referential_integrity(TABLES_CACHE)
    ALL_RESULTS.extend(ri_results)
    print(f"   ✅ {len(ri_results)} métricas de RI registradas")
except Exception as e:
    print(f"   ❌ ERRO RI: {str(e)[:200]}")
    TABLE_ERRORS.append({"table": "REFERENTIAL_INTEGRITY", "check": "ri", "error": str(e)[:500]})

EXECUTION_END = datetime.now(timezone.utc)
EXECUTION_DURATION = (EXECUTION_END - EXECUTION_START).total_seconds()

print(f"\n{'=' * 80}")
print(f"EXECUÇÃO CONCLUÍDA")
print(f"   Duração total: {EXECUTION_DURATION:.1f}s")
print(f"   Total de métricas: {len(ALL_RESULTS)}")
print(f"   Erros encontrados: {len(TABLE_ERRORS)}")
if TABLE_ERRORS:
    for err in TABLE_ERRORS:
        print(f"      ⚠️ {err.get('table', '?')}.{err.get('check', '?')}: {err['error'][:100]}")
print(f"{'=' * 80}")

In [0]:
# ============================================================================
# CÉLULA 4 — Cálculo do Quality Score por Tabela
# ============================================================================
# Fórmula transparente e documentada do Quality Score (0–100):
#
#   Score = (Completeness_Score * 0.30) + (Uniqueness_Score * 0.20) +
#          (Validity_Score * 0.20) + (Consistency_Score * 0.15) +
#          (Integrity_Score * 0.15)
#
# Onde cada sub-score é a média das métricas PASS/WARNING/FAIL da dimensão:
#   - PASS = 100, WARNING = 60, FAIL = 0
#
# Se uma dimensão não se aplica a uma tabela, seu peso é redistribuído
# proporcionalmente entre as dimensões aplicáveis.
#
# Status final:
#   Score >= 95 → PASS
#   80 <= Score < 95 → WARNING
#   Score < 80 → FAIL

# Mapeamento de status para valor numérico
STATUS_SCORES = {"PASS": 100.0, "WARNING": 60.0, "FAIL": 0.0}

# Pesos das dimensões
DIMENSION_WEIGHTS = {
    "Completeness": 0.30,
    "Uniqueness": 0.20,
    "Validity": 0.20,
    "Consistency": 0.15,
    "Integrity": 0.15,
}


def calculate_quality_score(table_full_name, all_results):
    """
    Calcula o Quality Score de uma tabela a partir das métricas registradas.
    Retorna: (score, status, dimension_scores_dict)
    """
    short_name = get_table_short_name(table_full_name)
    dimension_scores = {}

    # Agrupa resultados por dimensão (apenas para esta tabela)
    table_results = [r for r in all_results if r["table_name"] == table_full_name]

    # Para cada dimensão, calcula a média dos scores de status
    for dim in DIMENSION_WEIGHTS.keys():
        dim_results = [r for r in table_results if r["quality_dimension"] == dim]

        # Integrity pode ter resultados associados à tabela child
        if dim == "Integrity":
            dim_results = [r for r in all_results if r["quality_dimension"] == dim and r.get("child_table") == short_name]

        if not dim_results:
            continue  # Dimensão não se aplica

        # Filtra apenas resultados com status definido (exclui informational rows)
        status_results = [r for r in dim_results if r["status"] in STATUS_SCORES]
        if not status_results:
            continue

        dim_score = sum(STATUS_SCORES[r["status"]] for r in status_results) / len(status_results)
        dimension_scores[dim] = dim_score

    # Calcula score ponderado, redistribuindo pesos de dimensões ausentes
    total_weight = sum(DIMENSION_WEIGHTS[d] for d in dimension_scores.keys())
    if total_weight == 0:
        return 0.0, "FAIL", {}

    score = sum(dimension_scores[d] * (DIMENSION_WEIGHTS[d] / total_weight) for d in dimension_scores.keys())

    # Determina status final
    if score >= DQ_THRESHOLDS["pass"]:
        status = "PASS"
    elif score >= DQ_THRESHOLDS["warning"]:
        status = "WARNING"
    else:
        status = "FAIL"

    return round(score, 2), status, dimension_scores


# Calcula o Quality Score para cada tabela
QUALITY_SCORES = {}

print("=" * 80)
print("QUALITY SCORES POR TABELA")
print("=" * 80)
print(f"{'Tabela':<30} {'Score':>8} {'Status':>10}  {'Dimensões'}")
print(f"{'─' * 80}")

for table_full_name in BRONZE_TABLES:
    short_name = get_table_short_name(table_full_name)
    score, status, dim_scores = calculate_quality_score(table_full_name, ALL_RESULTS)
    QUALITY_SCORES[short_name] = {"score": score, "status": status, "dimensions": dim_scores}

    dim_str = " | ".join([f"{d[:4]}={v:.0f}" for d, v in dim_scores.items()])
    print(f"{short_name:<30} {score:>8.2f} {status:>10}  {dim_str}")

# Registra os scores como métricas no ALL_RESULTS
for table_full_name in BRONZE_TABLES:
    short_name = get_table_short_name(table_full_name)
    if short_name in QUALITY_SCORES:
        qs = QUALITY_SCORES[short_name]
        ALL_RESULTS.append(_make_base_row(
            table_full_name, "QualityScore", "quality_score", qs["score"],
            threshold=DQ_THRESHOLDS["pass"], status=qs["status"],
            rule_description=f"Score = weighted(Completeness*0.30, Uniqueness*0.20, Validity*0.20, Consistency*0.15, Integrity*0.15)"
        ))
        ALL_RESULTS.append(_make_base_row(
            table_full_name, "QualityScore", "quality_status", None,
            status=qs["status"],
            rule_description=f"Status: {qs['status']} (Score={qs['score']})"
        ))
        # Registra sub-scores por dimensão
        for dim_name, dim_score in qs["dimensions"].items():
            ALL_RESULTS.append(_make_base_row(
                table_full_name, "QualityScore", f"dimension_score_{dim_name}", dim_score,
                threshold=DIMENSION_THRESHOLDS.get(dim_name.lower(), {}).get("pass", 95.0),
                status="PASS" if dim_score >= 95 else ("WARNING" if dim_score >= 60 else "FAIL"),
                rule_description=f"{dim_name} dimension score"
            ))

print(f"\n{'=' * 80}")
print("✅ Quality Scores calculados e registrados!")

In [0]:
# ============================================================================
# CÉLULA 5 — Persistência Delta: data_quality + data_quality_summary + audit
# ============================================================================
# Converte ALL_RESULTS em DataFrame Spark e persiste em tabelas Delta.
# Usa mode("append") para preservar histórico de execuções anteriores.

DQ_TABLE = "credit_risk.bronze.data_quality"
DQ_SUMMARY_TABLE = "credit_risk.bronze.data_quality_summary"
DQ_AUDIT_TABLE = "credit_risk.bronze.data_quality_audit"

# ----------------------------------------------------------------------------
# 5.0 — Idempotência: remove dados de execuções anteriores do mesmo execution_id
# ----------------------------------------------------------------------------
# Garante que re-execuções do notebook não criem duplicatas para o mesmo execution_id.
for tbl in [DQ_TABLE, DQ_SUMMARY_TABLE, DQ_AUDIT_TABLE]:
    try:
        spark.sql(f"DELETE FROM {tbl} WHERE execution_id = '{EXECUTION_ID}'")
    except Exception:
        pass  # Tabela pode não existir ainda na primeira execução

# ----------------------------------------------------------------------------
# 5.1 — Criar/atualizar tabela detalhada: data_quality
# ----------------------------------------------------------------------------
print("📊 Persistindo resultados detalhados...")

if ALL_RESULTS:
    results_df = spark.createDataFrame(ALL_RESULTS)

    # Garante tipos consistentes para append
    results_df = results_df.withColumn("execution_timestamp", F.to_timestamp("execution_timestamp"))

    # Cria a tabela se não existir (na primeira execução)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DQ_TABLE} (
            execution_timestamp TIMESTAMP,
            execution_id STRING,
            batch_id STRING,
            table_name STRING,
            column_name STRING,
            quality_dimension STRING,
            metric_name STRING,
            metric_value DOUBLE,
            threshold DOUBLE,
            status STRING,
            rule_name STRING,
            rule_description STRING,
            parent_table STRING,
            child_table STRING,
            parent_key STRING,
            child_key STRING
        )
        USING DELTA
        COMMENT 'Data Quality detalhado — Credit Risk Bronze (histórico via append)'
    """)

    # Append preservando histórico
    results_df.write.mode("append").format("delta").saveAsTable(DQ_TABLE)
    print(f"   ✅ {len(ALL_RESULTS)} métricas persistidas em {DQ_TABLE}")
else:
    print("   ⚠️ Nenhuma métrica para persistir")

# ----------------------------------------------------------------------------
# 5.2 — Criar/atualizar tabela resumo: data_quality_summary
# ----------------------------------------------------------------------------
print("📊 Persistindo resumo por tabela...")

summary_rows = []
for table_full_name in BRONZE_TABLES:
    short_name = get_table_short_name(table_full_name)

    # Conta regras por status (excluindo métricas informativas)
    table_metrics = [r for r in ALL_RESULTS if r["table_name"] == table_full_name and r["status"] in ("PASS", "WARNING", "FAIL")]
    # Inclui RI onde a tabela é child
    ri_metrics = [r for r in ALL_RESULTS if r["quality_dimension"] == "Integrity" and r.get("child_table") == short_name and r["status"] in ("PASS", "WARNING", "FAIL")]
    all_table_metrics = table_metrics + ri_metrics

    total_rules = len(all_table_metrics)
    passed_rules = len([r for r in all_table_metrics if r["status"] == "PASS"])
    warning_rules = len([r for r in all_table_metrics if r["status"] == "WARNING"])
    failed_rules = len([r for r in all_table_metrics if r["status"] == "FAIL"])

    qs = QUALITY_SCORES.get(short_name, {"score": 0.0, "status": "FAIL"})

    summary_rows.append({
        "execution_timestamp": EXECUTION_TIMESTAMP,
        "execution_id": EXECUTION_ID,
        "batch_id": BATCH_ID,
        "table_name": short_name,
        "total_rules": total_rules,
        "passed_rules": passed_rules,
        "warning_rules": warning_rules,
        "failed_rules": failed_rules,
        "quality_score": float(qs["score"]),
        "overall_status": qs["status"],
    })

summary_df = spark.createDataFrame(summary_rows)
summary_df = summary_df.withColumn("execution_timestamp", F.to_timestamp("execution_timestamp"))

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DQ_SUMMARY_TABLE} (
        execution_timestamp TIMESTAMP,
        execution_id STRING,
        batch_id STRING,
        table_name STRING,
        total_rules INT,
        passed_rules INT,
        warning_rules INT,
        failed_rules INT,
        quality_score DOUBLE,
        overall_status STRING
    )
    USING DELTA
    COMMENT 'Data Quality resumo por tabela — Credit Risk Bronze (histórico via append)'
""")

summary_df.write.mode("append").format("delta").saveAsTable(DQ_SUMMARY_TABLE)
print(f"   ✅ {len(summary_rows)} resumos persistidos em {DQ_SUMMARY_TABLE}")

# ----------------------------------------------------------------------------
# 5.3 — Criar/atualizar tabela de auditoria: data_quality_audit
# ----------------------------------------------------------------------------
print("📊 Persistindo auditoria da execução...")

audit_rows = [{
    "execution_id": EXECUTION_ID,
    "batch_id": BATCH_ID,
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "start_time": EXECUTION_START.isoformat(),
    "end_time": EXECUTION_END.isoformat(),
    "duration_seconds": float(EXECUTION_DURATION),
    "execution_status": "COMPLETED_WITH_ERRORS" if TABLE_ERRORS else "SUCCESS",
    "tables_processed": len(BRONZE_TABLES),
    "tables_with_errors": len(TABLE_ERRORS),
    "total_metrics_generated": len(ALL_RESULTS),
    "error_details": "; ".join([f"{e.get('table', '?')}.{e.get('check', '?')}: {e['error'][:100]}" for e in TABLE_ERRORS]) if TABLE_ERRORS else "",
    "notebook_name": NOTEBOOK_NAME,
    "notebook_path": NOTEBOOK_PATH,
}]

audit_schema = StructType([
    StructField("execution_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("execution_timestamp", StringType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("execution_status", StringType(), True),
    StructField("tables_processed", IntegerType(), True),
    StructField("tables_with_errors", IntegerType(), True),
    StructField("total_metrics_generated", IntegerType(), True),
    StructField("error_details", StringType(), True),
    StructField("notebook_name", StringType(), True),
    StructField("notebook_path", StringType(), True),
])
audit_df = spark.createDataFrame(audit_rows, schema=audit_schema)
audit_df = audit_df.withColumn("execution_timestamp", F.to_timestamp("execution_timestamp"))

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DQ_AUDIT_TABLE} (
        execution_id STRING,
        batch_id STRING,
        execution_timestamp TIMESTAMP,
        start_time STRING,
        end_time STRING,
        duration_seconds DOUBLE,
        execution_status STRING,
        tables_processed INT,
        tables_with_errors INT,
        total_metrics_generated INT,
        error_details STRING,
        notebook_name STRING,
        notebook_path STRING
    )
    USING DELTA
    COMMENT 'Auditoria das execuções de Data Quality — Credit Risk Bronze'
""")

audit_df.write.mode("append").format("delta").saveAsTable(DQ_AUDIT_TABLE)
print(f"   ✅ Auditoria persistida em {DQ_AUDIT_TABLE}")

print(f"\n{'=' * 80}")
print("✅ PERSISTÊNCIA CONCLUÍDA!")
print(f"   Tabela detalhada: {DQ_TABLE}")
print(f"   Tabela resumo: {DQ_SUMMARY_TABLE}")
print(f"   Tabela auditoria: {DQ_AUDIT_TABLE}")
print(f"   Modo: append (histórico preservado)")
print(f"{'=' * 80}")

In [0]:
# ============================================================================
# CÉLULA 6 — Visualizações
# ============================================================================
# Outputs para visualização: qualidade por tabela, falhas, NULLs e TARGET.

print("\n" + "=" * 80)
print("VISUALIZAÇÕES — DATA QUALITY BRONZE")
print("=" * 80)

# ----------------------------------------------------------------------------
# 6.1 — Qualidade por tabela (score + status)
# ----------------------------------------------------------------------------
print("\n📊 6.1 — QUALITY SCORE POR TABELA")
print("─" * 60)

dq_summary = spark.sql(f"""
    SELECT table_name, quality_score, overall_status,
           total_rules, passed_rules, warning_rules, failed_rules
    FROM {DQ_SUMMARY_TABLE}
    WHERE execution_id = '{EXECUTION_ID}'
    ORDER BY quality_score DESC
""")
display(dq_summary)

# ----------------------------------------------------------------------------
# 6.2 — Falhas detalhadas (status = FAIL)
# ----------------------------------------------------------------------------
print("\n📊 6.2 — FALHAS DE DATA QUALITY (status = FAIL)")
print("─" * 60)

dq_failures = spark.sql(f"""
    SELECT table_name, column_name, quality_dimension,
           metric_name, metric_value, threshold,
           rule_name, rule_description
    FROM {DQ_TABLE}
    WHERE execution_id = '{EXECUTION_ID}'
      AND status = 'FAIL'
    ORDER BY table_name, quality_dimension
""")
fail_count = dq_failures.count()
print(f"Total de falhas: {fail_count}")
if fail_count > 0:
    display(dq_failures)
else:
    print("   ✅ Nenhuma falha registrada!")

# ----------------------------------------------------------------------------
# 6.3 — Warnings
# ----------------------------------------------------------------------------
print("\n📊 6.3 — WARNINGS DE DATA QUALITY")
print("─" * 60)

dq_warnings = spark.sql(f"""
    SELECT table_name, column_name, quality_dimension,
           metric_name, metric_value, threshold,
           rule_name, rule_description
    FROM {DQ_TABLE}
    WHERE execution_id = '{EXECUTION_ID}'
      AND status = 'WARNING'
    ORDER BY table_name, quality_dimension
""")
warning_count = dq_warnings.count()
print(f"Total de warnings: {warning_count}")
if warning_count > 0:
    display(dq_warnings.limit(100))
else:
    print("   ✅ Nenhum warning registrado!")

# ----------------------------------------------------------------------------
# 6.4 — NULLs — Top colunas com maior percentual de NULL
# ----------------------------------------------------------------------------
print("\n📊 6.4 — TOP 30 COLUNAS COM MAIOR PERCENTUAL DE NULL")
print("─" * 60)

dq_nulls = spark.sql(f"""
    SELECT table_name, column_name, metric_value AS null_percentage
    FROM {DQ_TABLE}
    WHERE execution_id = '{EXECUTION_ID}'
      AND metric_name = 'null_percentage'
      AND metric_value > 0
    ORDER BY metric_value DESC
    LIMIT 30
""")
display(dq_nulls)

# ----------------------------------------------------------------------------
# 6.5 — Distribuição do TARGET (application_train)
# ----------------------------------------------------------------------------
print("\n📊 6.5 — DISTRIBUIÇÃO DA VARIÁVEL TARGET (application_train)")
print("─" * 60)

target_dist = spark.sql(f"""
    SELECT rule_name, metric_name, metric_value
    FROM {DQ_TABLE}
    WHERE execution_id = '{EXECUTION_ID}'
      AND table_name = 'credit_risk.bronze.application_train'
      AND quality_dimension = 'TargetDistribution'
    ORDER BY rule_name, metric_name
""")
display(target_dist)

# Também exibe a distribuição direto da tabela
print("\nDistribuição direta:")
target_direct = spark.table("credit_risk.bronze.application_train").groupBy("TARGET").count().orderBy("TARGET")
display(target_direct)

In [0]:
# ============================================================================
# CÉLULA 7 — Histórico de Execuções + Resumo Final
# ============================================================================

# ----------------------------------------------------------------------------
# 7.1 — Histórico de execuções (comparação de scores entre execuções)
# ----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("HISTÓRICO DE EXECUÇÕES — DATA QUALITY SCORE POR TABELA")
print("=" * 80)

history_df = spark.sql(f"""
    SELECT execution_timestamp,
           table_name,
           quality_score,
           overall_status,
           total_rules,
           passed_rules,
           warning_rules,
           failed_rules
    FROM {DQ_SUMMARY_TABLE}
    ORDER BY execution_timestamp ASC, table_name
""")
display(history_df)

# ----------------------------------------------------------------------------
# 7.2 — Comparação de scores entre as últimas 2 execuções
# ----------------------------------------------------------------------------
print("\n📈 COMPARAÇÃO ENTRE EXECUÇÕES (últimas 2)")
print("─" * 60)

trend_df = spark.sql(f"""
    WITH ranked AS (
        SELECT table_name, quality_score, execution_timestamp,
               ROW_NUMBER() OVER (PARTITION BY table_name ORDER BY execution_timestamp DESC) AS rn
        FROM {DQ_SUMMARY_TABLE}
    )
    SELECT
        a.table_name,
        ROUND(a.quality_score, 2) AS latest_score,
        a.execution_timestamp AS latest_execution,
        ROUND(b.quality_score, 2) AS previous_score,
        b.execution_timestamp AS previous_execution,
        ROUND(a.quality_score - b.quality_score, 2) AS score_delta
    FROM ranked a
    JOIN ranked b ON a.table_name = b.table_name AND b.rn = 2
    WHERE a.rn = 1
    ORDER BY a.table_name
""")
if trend_df.count() > 0:
    display(trend_df)
else:
    print("   (Apenas 1 execução registrada — execute novamente para comparar)")

# ----------------------------------------------------------------------------
# 7.3 — Auditoria da execução atual
# ----------------------------------------------------------------------------
print("\n📋 AUDITORIA DA EXECUÇÃO ATUAL")
print("─" * 60)
audit_display = spark.sql(f"""
    SELECT execution_id, execution_status, duration_seconds,
           tables_processed, tables_with_errors, total_metrics_generated,
           error_details
    FROM {DQ_AUDIT_TABLE}
    WHERE execution_id = '{EXECUTION_ID}'
""")
display(audit_display)

# ----------------------------------------------------------------------------
# 7.4 — Resumo Final do Notebook
# ----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("RESUMO FINAL — 04_data_quality_bronze")
print("=" * 80)

# Conta métricas por dimensão
dimension_counts = {}
for r in ALL_RESULTS:
    dim = r["quality_dimension"]
    dimension_counts[dim] = dimension_counts.get(dim, 0) + 1

# Conta status gerais
status_counts = {"PASS": 0, "WARNING": 0, "FAIL": 0}
for r in ALL_RESULTS:
    if r["status"] in status_counts:
        status_counts[r["status"]] += 1

# Tabela com pior score
worst_table = min(QUALITY_SCORES.items(), key=lambda x: x[1]["score"])
best_table = max(QUALITY_SCORES.items(), key=lambda x: x[1]["score"])

print(f"\n1. O QUE FOI IMPLEMENTADO:")
print(f"   • 9 dimensões de Data Quality: Completeness, Uniqueness, Validity (Type+Domain),")
print(f"     Consistency (Business Rules), Integrity (Referential), Outliers, Numeric Stats,")
print(f"     Target Distribution, Quality Score")
print(f"   • 8 tabelas Bronze analisadas")
print(f"   • {len(ALL_RESULTS)} métricas geradas no total")
print(f"   • 3 tabelas Delta de persistência: data_quality, data_quality_summary, data_quality_audit")

print(f"\n2. REGRAS CRIADAS:")
print(f"   • {sum(len(v) for v in BUSINESS_RULES.values())} regras de negócio (Consistency)")
print(f"   • {len(EXPECTED_DOMAINS)} validações de domínio (Validity)")
print(f"   • {len(REFERENTIAL_INTEGRITY_MAPPINGS)} verificações de integridade referencial (Integrity)")
print(f"   • Detecção de outliers (IQR) para {sum(len(v) for v in NUMERIC_PRIORITY_COLUMNS.values())} colunas numéricas")

print(f"\n3. MÉTRICAS CALCULADAS POR DIMENSÃO:")
for dim, count in sorted(dimension_counts.items()):
    print(f"   • {dim}: {count} métricas")

print(f"\n4. TABELAS AVALIADAS:")
for table_full_name in BRONZE_TABLES:
    sn = get_table_short_name(table_full_name)
    qs = QUALITY_SCORES.get(sn, {"score": 0, "status": "N/A"})
    print(f"   • {sn}: Score={qs['score']:.2f} Status={qs['status']}")

print(f"\n5. RESULTADOS DAS REGRAS:")
print(f"   • PASS: {status_counts['PASS']}")
print(f"   • WARNING: {status_counts['WARNING']}")
print(f"   • FAIL: {status_counts['FAIL']}")

print(f"\n6. TABELA COM PRINCIPAIS PROBLEMAS:")
print(f"   • Pior score: {worst_table[0]} ({worst_table[1]['score']:.2f} — {worst_table[1]['status']})")
print(f"   • Melhor score: {best_table[0]} ({best_table[1]['score']:.2f} — {best_table[1]['status']})")

print(f"\n7. PONTOS PARA TRATAMENTO NA SILVER:")
print(f"   • Tratar NULLs identificados (especialmente em colunas com alto percentual)")
print(f"   • Investigar e tratar duplicidades em chaves candidatas")
print(f"   • Tratar outliers detectados via IQR (capping, remoção ou transformação)")
print(f"   • Avaliar valores fora do domínio em colunas categóricas")
print(f"   • Validar integridade referencial — órfãos podem indicar dados faltantes")
print(f"   • Considerar balanceamento da variável TARGET (se desbalanceada)")
print(f"   • Converter colunas DAYS_* para valores positivos (anos/dias)")
print(f"   • Corrigir tipos de dados quando necessário")

print(f"\n8. MELHORIAS FUTURAS DE DATA QUALITY:")
print(f"   • Adicionar regras de cross-table validation (ex: AMT_CREDIT em previous_application vs application_train)")
print(f"   • Implementar drift detection entre execuções")
print(f"   • Adicionar validação de formato (datas, emails, CPF/CNPJ)")
print(f"   • Implementar data profiling automatizado para novas colunas")
print(f"   • Criar alertas automáticos quando score cair abaixo do threshold")
print(f"   • Adicionar validação de cardinalidade (low/high cardinality detection)")
print(f"   • Implementar checks de fresness (dados atualizados dentro do prazo)")

print(f"\n{'=' * 80}")
print(f"✅ NOTEBOOK 04_data_quality_bronze — EXECUÇÃO CONCLUÍDA")
print(f"   Execution ID: {EXECUTION_ID}")
print(f"   Duração: {EXECUTION_DURATION:.1f}s")
print(f"   Métricas: {len(ALL_RESULTS)}")
print(f"   Tabelas com erro: {len(TABLE_ERRORS)}")
print(f"{'=' * 80}")